# An RNN is an iterated block map

This notebook shows that a `torch.nn.RNN` is one point in a much larger family
of models, by rebuilding it as a *block map iterated as a dynamical system* and
checking we get the same numbers back.

The whole module is one equation. Writing $z$ for the state, split into slots,

$$z_{t+1} = \underbrace{(A \circ b \circ M)^{K}}_{\text{fast, internal}} \circ
\underbrace{\mathrm{Inject}_{t+1}}_{\text{slow, external}} \circ\; z_t$$

| symbol | what it is |
| --- | --- |
| $M$ | a `Sequential2D`: a matrix whose entries are (possibly nonlinear) maps |
| $b$ | a bias, one vector per slot |
| $A$ | an activation, one module per slot |
| $\mathrm{Inject}$ | overwrite the input slot with the next token |
| $K$ | how many times the internal map runs per token |

Read right to left: a token arrives and is written into the input slot; the
internal map runs $K$ times; the next token arrives.

**The punchline.** $K = 1$ *is* an ordinary RNN. $K > 1$ is not, and is the
reason the module exists — it lets the network run its own dynamics at a
different rate from the sequence's clock.

Background: Hershey, Paffenroth, Pathak & Tavener,
[arXiv:2404.00880](https://arxiv.org/abs/2404.00880). Design decisions and
their justifications are in `OVERVIEW_RNN_SEQUENTIAL_2D.md`.

In [8]:
import torch

from iterativennsimple.Sequential2D import Sequential2D, Identity
from iterativennsimple.Sequential2DRNN import Sequential2DRNN

torch.manual_seed(0)

## 1. What we are trying to reproduce

PyTorch's RNN is the recurrence

$$h_t = \tanh\!\big(W_{ih}\,x_t + b_{ih} + W_{hh}\,h_{t-1} + b_{hh}\big)$$

Small enough to inspect by hand:

In [9]:
input_size, hidden_size, batch_size, seq_len = 3, 4, 2, 5

rnn = torch.nn.RNN(input_size, hidden_size, num_layers=1,
                   nonlinearity='tanh', batch_first=True)

x = torch.randn(batch_size, seq_len, input_size)
h_0 = torch.randn(1, batch_size, hidden_size)

output_ref, h_n_ref = rnn(x, h_0)
print(f'output {tuple(output_ref.shape)}   h_n {tuple(h_n_ref.shape)}')

output (2, 5, 4)   h_n (1, 2, 4)


## 2. The same thing as a block map

Take the state to be two slots, $z = [x, h]$. Then the recurrence is the block
matrix (rows are outputs, the usual mathematics convention)

$$M = \begin{bmatrix} I & \texttt{None} \\ W_{xh} & W_{hh} \end{bmatrix}
\qquad b = [\,0,\; b_{ih} + b_{hh}\,] \qquad A = [\,\mathrm{id},\; \tanh\,]$$

Applying $M$ to $[x_{t+1}, h_t]$ gives $[x_{t+1},\; W_{xh}x_{t+1} + W_{hh}h_t]$,
and then $b$ and $A$ turn the second slot into
$\tanh(W_{xh}x_{t+1} + W_{hh}h_t + b)$ — the recurrence above.

Three things are worth pausing on.

### (a) The activation goes on the slot, not on the blocks

$A$ is applied *after* everything arriving at a slot has been summed. That is
what gives

$$\tanh(W_{xh}x + W_{hh}h) \qquad\text{and not}\qquad \tanh(W_{xh}x) + \tanh(W_{hh}h)$$

These are different functions, and only the first is an RNN. So the activation
cannot live inside the individual blocks.

### (b) `blocks[i][j]` is indexed (input, output) — the transpose of the matrix

This is the single easiest thing to get wrong. In `Sequential2D`,
`blocks[i][j]` takes slot `i` and *produces* slot `j`. Written as a matrix,
rows are outputs. So the code array is the transpose of the matrix on the page:

| in the matrix | means | in the code |
| --- | --- | --- |
| $M_{x,x} = I$ | $x \to x$ | `blocks[0][0]` |
| $M_{h,x} = W_{xh}$ | $x \to h$ | `blocks[0][1]` |
| $M_{h,h} = W_{hh}$ | $h \to h$ | `blocks[1][1]` |

A reassuring fact that removes a related worry: PyTorch stores weights in the
*mathematics* convention. `torch.nn.Linear(i, o).weight` has shape `(o, i)`,
row = output, and the transpose happens inside `forward`. So `weight_ih_l0` is
literally $W_{ih}$ and **the weights below copy across with no transpose.**

### (c) One bias per slot, not two

PyTorch carries `bias_ih` and `bias_hh`. Only their sum is ever visible to the
forward pass, so we store one bias per slot and set $b_h = b_{ih} + b_{hh}$.
The outputs agree exactly; the map back to a PyTorch `state_dict` is just not
unique.

## 3. Build it by hand

`Sequential2DRNN.from_rnn` does all of this for you, but doing it once
explicitly is the point of this notebook.

In [10]:
W_xh = torch.nn.Linear(input_size, hidden_size, bias=False)   # bias lives on the slot
W_hh = torch.nn.Linear(hidden_size, hidden_size, bias=False)

with torch.no_grad():
    W_xh.weight.copy_(rnn.weight_ih_l0)     # (hidden, input) -> (hidden, input), no transpose
    W_hh.weight.copy_(rnn.weight_hh_l0)     # (hidden, hidden)
    b_h = rnn.bias_ih_l0 + rnn.bias_hh_l0   # two PyTorch biases -> one slot bias

# blocks[i][j] maps slot i to slot j; slots are x = 0, h = 1.
blocks = [[Identity(in_features=input_size, out_features=input_size), W_xh],
          [None,                                                      W_hh]]

by_hand = Sequential2DRNN(
    features_list=[input_size, hidden_size],
    blocks=blocks,
    bias=[None, b_h],
    activation=[torch.nn.Identity(), torch.nn.Tanh()],
    inject_slot=0, hidden_slot=1, output_slot=1,
    K=1, batch_first=True,
)

output, h_n = by_hand(x, h_0)

print(f'largest disagreement in output: {(output - output_ref).abs().max():.3e}')
print(f'largest disagreement in h_n   : {(h_n - h_n_ref).abs().max():.3e}')

assert torch.allclose(output, output_ref, atol=1e-6)
assert torch.allclose(h_n, h_n_ref, atol=1e-6)

largest disagreement in output: 5.960e-08
largest disagreement in h_n   : 5.960e-08


Same answer to floating-point tolerance. The factory does exactly the above:

In [11]:
from_factory, _ = Sequential2DRNN.from_rnn(rnn)(x, h_0)
assert torch.allclose(from_factory, output_ref, atol=1e-6)
print('from_rnn agrees too')

from_rnn agrees too


## 4. Looking inside one step

Because the state is just a list of slots, you can drive the system by hand and
look at every intermediate quantity. Here is one external step, unpacked:

In [12]:
z = [None, h_0[0]]                       # [x-slot not yet alive, h_0]
x_1 = x[:, 0, :]                         # the first token

z_injected = list(z)
z_injected[0] = x_1                      # Inject
z_next = by_hand.internal_step(z_injected)   # A . b . M

h_1_by_hand = torch.tanh(x_1 @ rnn.weight_ih_l0.T + h_0[0] @ rnn.weight_hh_l0.T + b_h)

print(f'from the map      : {z_next[1][0].detach().numpy().round(4)}')
print(f'from the formula  : {h_1_by_hand[0].detach().numpy().round(4)}')
assert torch.allclose(z_next[1], h_1_by_hand, atol=1e-6)

from the map      : [-0.9252 -0.0057  0.377  -0.7421]
from the formula  : [-0.9252 -0.0057  0.377  -0.7421]


Note `z` started with `None` in the input slot. `None` means *not yet alive* —
nothing has written to that slot — and is deliberately **not** the same as the
slot holding a zero vector, since $A(0 + b)$ is generally neither zero nor
`None`. `None` also lets `Sequential2D` skip work, which is where the
efficiency of sparse block structures comes from.

## 5. The input slot is a skip connection

The $I$ in the corner of $M$ holds the injected token unchanged while the
internal map runs. At $K = 1$ that does nothing — `Inject` overwrites the slot
anyway. At $K > 1$ it means every internal iteration sees the same forcing:

In [13]:
z = [x_1, None, torch.zeros(batch_size, hidden_size)]
three_slot = Sequential2DRNN.from_3x3(
    input_size, 2, hidden_size,
    W_xh=torch.nn.Linear(input_size, hidden_size, bias=False),
    W_hh=torch.nn.Linear(hidden_size, hidden_size, bias=False),
    W_hy=torch.nn.Linear(hidden_size, 2, bias=False),
    batch_first=True)

for k in range(4):
    z = three_slot.internal_step(z)
    print(f'internal iteration {k+1}: x-slot unchanged = {torch.equal(z[0], x_1)}')

internal iteration 1: x-slot unchanged = True
internal iteration 2: x-slot unchanged = True
internal iteration 3: x-slot unchanged = True
internal iteration 4: x-slot unchanged = True


Unroll the $K$ internal iterations as a $K$-layer network and this is exactly a
skip connection: the input reaches every layer without passing through any
$W_{hh}$ or nonlinearity, so there is always a depth-1 gradient path back to
it. Without it, the input would enter only at iteration 1 and its gradient
would have to survive $K$ layers.

It is an *input* skip, though, not a ResNet-style *state* residual — the
hidden recursion's own Jacobian is untouched, so it does nothing for vanishing
gradients in $h$.

## 6. Turning the dial: $K > 1$

Same weights, more internal iterations per token. This is no longer an RNN.

In [14]:
for K in [1, 2, 5]:
    model = Sequential2DRNN.from_rnn(rnn, K=K)
    out, _ = model(x, h_0)
    drift = (out - output_ref).abs().max()
    print(f'K = {K}:  |output - RNN output|_max = {drift:.4f}')

K = 1:  |output - RNN output|_max = 0.0000
K = 2:  |output - RNN output|_max = 1.4739
K = 5:  |output - RNN output|_max = 0.9846


As $K$ grows the state has time to relax toward a fixed point of the internal
map *between* tokens. Since the input slot holds $x_{t+1}$ throughout, that
fixed point depends on the input — it solves $z^\star = A(b + M z^\star)$,
which is what a deep equilibrium model computes. Watch the state settle:

In [15]:
z = [x_1, torch.zeros(batch_size, hidden_size)]
previous = z[1]
for k in range(8):
    z = by_hand.internal_step(z)
    print(f'iteration {k+1}: ||h_k - h_(k-1)|| = {(z[1] - previous).norm():.6f}')
    previous = z[1]

iteration 1: ||h_k - h_(k-1)|| = 1.622271
iteration 2: ||h_k - h_(k-1)|| = 0.609969
iteration 3: ||h_k - h_(k-1)|| = 0.255317
iteration 4: ||h_k - h_(k-1)|| = 0.082911
iteration 5: ||h_k - h_(k-1)|| = 0.046400
iteration 6: ||h_k - h_(k-1)|| = 0.024857
iteration 7: ||h_k - h_(k-1)|| = 0.013711
iteration 8: ||h_k - h_(k-1)|| = 0.007529


Whether it settles at all depends on the weights: this is now a question about
the spectrum of the map, not about neural networks. That is the point of the
whole reformulation — familiar dynamical-systems questions become the right
questions to ask about the architecture.

## 7. The general three-slot map

`from_3x3` opens up a state $z = [x, y, h]$ with six free blocks — the $y$ and
$h$ rows of the matrix:

$$M = \begin{bmatrix}
I & \texttt{None} & \texttt{None} \\
W_{xy} & W_{yy} & W_{hy} \\
W_{xh} & W_{yh} & W_{hh}
\end{bmatrix}$$

Naming is source-first: $W_{ab}$ maps slot $a$ to slot $b$. Blocks may be
`Linear`, `MaskedLinear`, `MonarchLinear`, a `Sequential1D`, or a nested
`Sequential2D` — anything carrying `in_features` and `out_features`. `None`
means the block is absent.

One subtlety to internalise: `Sequential2D` applies a **Jacobi** update, so
every block reads the *old* state. A readout slot is therefore always one
internal iteration behind. With $W_{hy} = I$ you can see it directly:

In [16]:
lagging = Sequential2DRNN.from_3x3(
    input_size, hidden_size, hidden_size,
    W_xh=torch.nn.Linear(input_size, hidden_size, bias=False),
    W_hh=torch.nn.Linear(hidden_size, hidden_size, bias=False),
    W_hy=Identity(in_features=hidden_size, out_features=hidden_size),
    A_y=torch.nn.Identity(), bias=False, batch_first=True)

h_before = torch.randn(batch_size, hidden_size)
z = lagging.internal_step([x_1, None, h_before])

print(f'y-slot equals the *previous* h: {torch.equal(z[1], h_before)}')
print(f'y-slot equals the *new* h     : {torch.equal(z[1], z[2])}')

y-slot equals the *previous* h: True
y-slot equals the *new* h     : False


So if you want `output` to be $h_t$ itself — which is what `torch.nn.RNN`
returns — read the $h$-slot, which is what `from_rnn` sets up. Read the
$y$-slot only when you want a readout, and remember it trails by one.

## Where to go next

- `OVERVIEW_RNN_SEQUENTIAL_2D.md` — the derivation and every design decision,
  including the ones deliberately left out (multi-layer stacking, bidirectional,
  `PackedSequence`) and why.
- `examples/rnn_internal_iterations.py` — trains at several values of $K$ on a
  task that needs memory, and asks whether internal iterations can substitute
  for sequence-length memory.
- `tests/test_Sequential2DRNN.py` — the invariants, most of which fail silently
  if broken.

Open questions, if you are looking for something to work on: what does the
spectrum of $M$ predict about trainability, and does a sparse $W_{hh}$
(`MonarchLinear`, `MaskedLinear`) with larger $K$ beat a dense one at $K = 1$
with the same parameter count?